# CEDA 토론 시스템 테스트

AI 에이전트들이 CEDA 형식으로 토론하는 시스템을 테스트합니다.

- 10개 토론 턴 + 1 심판 판정 = 총 11턴
- 긍정측/부정측 State 격리
- 측별 모델 지정 가능

## 1. Import 및 환경 확인

In [6]:
import sys
sys.path.insert(0, '../..')

from Agent_Structure.debate import (
    run_debate,
    stream_debate,
    create_debate,
    DebateResult,
    DebateState,
    CEDA_ROUNDS,
)

print(f"CEDA 라운드: {len(CEDA_ROUNDS)}개")
for i, r in enumerate(CEDA_ROUNDS, 1):
    print(f"  {i}. [{r['round_id']}] {r['speaker']} - {r['speech_type']}")

CEDA 라운드: 10개
  1. [1AC] affirmative - constructive
  2. [CX_1AC_Q] negative - cx_question
  3. [CX_1AC_A] affirmative - cx_answer
  4. [1NC] negative - constructive
  5. [CX_1NC_Q] affirmative - cx_question
  6. [CX_1NC_A] negative - cx_answer
  7. [1AR] affirmative - rebuttal
  8. [1NR] negative - rebuttal
  9. [2NR] negative - rebuttal
  10. [2AR] affirmative - rebuttal


In [7]:
# 환경변수 확인
from Agent_Structure.config import settings

print(f"Provider: {settings.default_provider}")
print(f"Model: {settings.default_model}")
print(f"API Key 설정됨: {bool(settings.anthropic_api_key)}")

Provider: anthropic
Model: claude-sonnet-4-5-20250929
API Key 설정됨: True


## 2. 스트리밍 토론 실행

라운드별로 결과를 확인할 수 있어 진행 상황을 추적하기 좋습니다.

In [8]:
PROPOSITION = "남녀 사이에 진정한 우정은 가능한가?"

speaker_labels = {"affirmative": "긍정측", "negative": "부정측", "judge": "심판"}
type_labels = {
    "constructive": "입론",
    "cx_question": "교차조사 질문",
    "cx_answer": "교차조사 답변",
    "rebuttal": "반박",
    "verdict": "판정",
}

speeches = []

for speech in stream_debate(PROPOSITION):
    speeches.append(speech)
    speaker = speaker_labels.get(speech['speaker'], speech['speaker'])
    stype = type_labels.get(speech['speech_type'], speech['speech_type'])
    
    print(f"\n{'='*60}")
    print(f"[{speech['round_id']}] {speaker} — {stype}")
    print(f"{'='*60}")
    print(speech['content'])

print(f"\n\n총 {len(speeches)}개 발언 완료")


[1AC] 긍정측 — 입론
# 1AC 입론 (긍정측)

존경하는 심판님, 그리고 부정측 토론자 여러분. 저는 오늘 "남녀 사이에 진정한 우정은 가능하다"는 논제를 입증하겠습니다.

## 정의(Definition)

먼저 핵심 용어를 정의하겠습니다.
- **진정한 우정**: 상호 존중과 신뢰를 바탕으로 한 비낭만적(non-romantic) 관계로서, 서로의 행복을 진심으로 바라는 관계
- **가능하다**: 현실에서 실제로 존재하며, 지속 가능한 형태로 유지될 수 있음

## Contention 1: 우정의 본질은 성별과 무관하다

**주장**: 우정을 구성하는 핵심 요소들은 성별과 독립적입니다.

**이유**: 우정의 기본 조건은 공통의 관심사, 상호 존중, 신뢰, 감정적 지지입니다. 이러한 요소들은 생물학적 성별에 의해 결정되지 않습니다.

**근거**:
1. **철학적 근거**: 아리스토텔레스는 『니코마코스 윤리학』에서 진정한 우정(philia)을 "상대방의 덕을 인정하고 그 자체로 존중하는 관계"로 정의했습니다. 이 정의에는 성별 조건이 없습니다.

2. **심리학적 증거**: 2012년 위스콘신 대학 연구에 따르면, 깊은 우정을 형성하는 심리적 메커니즘(공감 능력, 자기 개방, 상호성)은 남녀 관계와 동성 관계에서 동일하게 작동합니다.

3. **현실 사례**: 전문직 환경에서 남녀 멘토-멘티 관계는 낭만적 요소 없이 수년간 지속되며, 서로의 성장을 돕는 진정한 우정으로 발전합니다.

## Contention 2: 성적 끌림과 우정은 양립 가능하다

**주장**: 일시적인 성적 끌림이 존재한다고 해서 진정한 우정이 불가능해지는 것은 아닙니다.

**이유**: 인간은 복합적인 감정을 동시에 경험하고 관리할 수 있는 이성적 존재입니다.

**근거**:
1. **인지심리학적 증거**: 인간은 감정과 행동 사이에 이성적 판단을 개입시킬 수 있습니다. 2015년 노르웨이 베르겐 대학 연구는 친구 관계에서 일시적 끌림을 경험하더라도 85%가 우정을 유지하는 것을 

## 3. 일괄 실행 (run_debate)

전체 토론을 한 번에 실행하고 `DebateResult`로 받습니다.

In [ ]:
result = run_debate("인공지능 개발에 대한 국제적 규제가 필요하다")

print(f"논제: {result.proposition}")
print(f"총 발언 수: {len(result.transcript)}")
print(f"\n{'='*60}")
print("전체 토론 기록")
print(f"{'='*60}")
print(result.format_transcript())
print(f"\n{'='*60}")
print("최종 판정")
print(f"{'='*60}")
print(result.verdict)

## 4. 측별 모델 지정 (모델 대결)

긍정측과 부정측에 서로 다른 LLM을 사용할 수 있습니다.

In [ ]:
# 예시: 같은 프로바이더 내에서 모델만 변경
# OpenAI API 키가 있는 경우 아래 주석을 해제하세요

# result = run_debate(
#     "인공지능 개발에 대한 국제적 규제가 필요하다",
#     aff_provider_name="anthropic",
#     aff_model_name="claude-sonnet-4-5-20250929",
#     neg_provider_name="openai",
#     neg_model_name="gpt-4o",
# )
# print(result.format_transcript())

## 5. 도구 포함 토론

토론 에이전트에게 웹 검색 등 도구를 제공할 수 있습니다.

In [ ]:
# 도구 포함 토론 (TAVILY_API_KEY 필요)

# from Agent_Structure.tools import tool_registry
# search_tools = tool_registry.get_by_tag("search")
# print(f"사용 가능한 검색 도구: {[t.name for t in search_tools]}")

# result = run_debate(
#     "원자력 발전소를 확대해야 한다",
#     tools=search_tools,
# )
# print(result.format_transcript())

## 6. 그래프 구조 확인

In [ ]:
graph, initial_state = create_debate("테스트 논제")

print("초기 상태:")
for k, v in initial_state.items():
    if k == 'round_sequence':
        print(f"  {k}: {len(v)}개 라운드")
    else:
        print(f"  {k}: {repr(v)[:80]}")

print(f"\n그래프 노드: {graph.get_graph().nodes}")